# 08. SELECT · WHERE · Predicates

*Module: 3 — DML · Companion summary: `lectures/08-select-statement-where-clause.md` · SQL engine: SQLite (in-memory)*

## What you will study

- retrieve all rows / a subset of columns / a restricted set of rows
- define predicate (true / false / unknown) and use it in WHERE
- use all six comparison operators

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — the BOOK table from the course

In [ ]:
q("CREATE TABLE book (book_id TEXT PRIMARY KEY, title TEXT, pages INTEGER, year INTEGER)")
q("""
INSERT INTO book VALUES
    ('B1', 'The Little Prince',   292, 1943),
    ('B2', 'Journey to the West', 298, 1592),
    ('B3', 'SQL Basics',          150, 2018),
    ('B4', 'Data Science Handbook', 400, 2020),
    ('B5', 'A Short Story',        45, 2022)
""")
q("SELECT * FROM book")

## 🧪 Choose your columns — output order follows the SELECT list

In [ ]:
q("SELECT book_id, title FROM book")

## 🧪 WHERE with a predicate — restrict to rows where it evaluates TRUE

In [ ]:
q("SELECT book_id, title FROM book WHERE book_id = 'B1'")
q("SELECT title, pages FROM book WHERE pages > 200")

## 🧪 All six comparison operators, one query each

In [ ]:
print('equal to            ='); q("SELECT title FROM book WHERE book_id = 'B2'")
print('greater than        >'); q("SELECT title FROM book WHERE pages > 290")
print('less than           <'); q("SELECT title FROM book WHERE pages < 150")
print('greater or equal   >='); q("SELECT title FROM book WHERE pages >= 292")
print('less or equal      <='); q("SELECT title FROM book WHERE pages <= 292")
print('not equal           <>'); q("SELECT title FROM book WHERE book_id <> 'B1'")

## 🧪 Combine predicates with AND / OR

In [ ]:
q("SELECT title FROM book WHERE pages > 200 AND year > 2018")
q("SELECT title FROM book WHERE year = 1943 OR pages = 45")

## 💡 Tips & tricks

- Avoid `SELECT *` in production code: name the columns (less traffic, self-documenting,
  survives schema changes).
- 'B1' compares strings, 292 compares numbers — quotes matter.

## 🎯 Your turn

1. Titles of books with `pages >= 150` AND `year >= 2018`.
2. All columns of books that are NOT 'B3'.

In [ ]:
# 💡 Solution
q("SELECT title FROM book WHERE pages >= 150 AND year >= 2018")
q("SELECT * FROM book WHERE book_id <> 'B3'")